In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import torch
from src.config import DATASET_ROOT, POSE_DATASET_ROOT, GAMMA_POSE_DATASET_ROOT
from src.Skeleton_model.yolo_pose_tracking import save_annotated_pose_videos
from src.rwf2000 import RWF2000PoseDataset 
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.Skeleton_model.stgcn import STGCN
from scripts.common.get_device import get_available_device
from ultralytics import YOLO
from pathlib import Path
import torch
import numpy as np
from src.Skeleton_model.yolo_pose_tracking import pose_data_to_stgcn_tensor

In [3]:
pose_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train")
radii = compute_joint_distance_to_center_of_gravity(pose_dataset)
skeleton_graph = SkeletonGraph(radii)
device = get_available_device()
model = STGCN(adjacency=skeleton_graph.A).to(device)

/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


Using cuda:0 with 20.55 GB free


In [26]:
pose_path, label = pose_dataset.samples[10]
pose_data = torch.load(pose_path, weights_only=False)
print(pose_data["frames"][140]['people'][0]['keypoint_confidence'].sum())

tensor(13.9065)


In [ ]:
from collections import defaultdict
def pose_data_to_stgcn_tensor(
    pose_data,
    num_frames=150,
    num_keypoints=17,
    max_people=2,
):
    """
    Convert pose tracking data to ST-GCN tensor.

    Output shape:
        [C, T, V, M]

    where:
        C = (normalised x, normalised y, confidence)
        T = number of frames
        V = number of keypoints
        M = maximum number of tracked people
    """

    H, W = pose_data["original_video_shape"]

    # 1. Score each track by cumulative keypoint confidence.
    # This favours tracks that are both long-lasting and consistently
    # detected with high confidence.
    track_scores = defaultdict(float)

    for frame in pose_data["frames"]:
        for person in frame["people"]:
            track_id = int(person["track_id"])
            confidences = person["keypoint_confidence"].float()

            track_scores[track_id] += confidences.sum().item()

    # 2. Select the strongest tracks
    sorted_tracks = sorted(
        track_scores.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    selected_track_ids = [
        track_id
        for track_id, score in sorted_tracks[:max_people]
    ]

    # Map each selected track to a fixed person dimension
    track_to_person_index = {
        track_id: person_index
        for person_index, track_id in enumerate(selected_track_ids)
    }

    # 3. Construct output tensor
    tensor = torch.zeros(
        3,
        num_frames,
        num_keypoints,
        max_people,
        dtype=torch.float32,
    )

    # 4. Fill tensor with pose information
    for frame in pose_data["frames"]:
        frame_index = frame["frame_index"]

        if not 0 <= frame_index < num_frames:
            continue

        for person in frame["people"]:
            track_id = int(person["track_id"])

            if track_id not in track_to_person_index:
                continue

            person_index = track_to_person_index[track_id]

            keypoints = person["keypoints"].float()
            confidences = person["keypoint_confidence"].float()

            # Normalise coordinates to [0, 1]
            x = keypoints[:, 0] / W
            y = keypoints[:, 1] / H

            tensor[0, frame_index, :, person_index] = x
            tensor[1, frame_index, :, person_index] = y
            tensor[2, frame_index, :, person_index] = confidences

    return tensor

In [2]:
from pathlib import Path
import torch
import numpy as np

def analyse_pose_database(pose_root):
    pose_files = list(Path(pose_root).rglob("*.pt"))

    empty_samples = 0
    detection_coverages = []
    tracking_coverages = []
    visible_joints_per_frame = []
    keypoint_confidences = []
    unique_track_ids_per_video = []

    for pose_path in pose_files:
        pose_data = torch.load(pose_path, weights_only=False)
        frames = pose_data["frames"]

        frames_with_detections = 0
        frames_with_tracks = 0
        total_visible_joints = 0
        track_ids = set()
        sample_has_pose = False

        for frame in frames:
            people = frame["people"]

            if len(people) > 0:
                frames_with_detections += 1
                sample_has_pose = True

            frame_has_track = False

            for person in people:
                if person["track_id"] is not None:
                    track_ids.add(person["track_id"])
                    frame_has_track = True

                confidence = person["keypoint_confidence"]
                visible = confidence > 0

                total_visible_joints += visible.sum().item()
                keypoint_confidences.extend(
                    confidence[visible].tolist()
                )

            if frame_has_track:
                frames_with_tracks += 1

        num_frames = len(frames)

        if not sample_has_pose:
            empty_samples += 1

        detection_coverages.append(
            frames_with_detections / num_frames
        )

        tracking_coverages.append(
            frames_with_tracks / num_frames
        )

        visible_joints_per_frame.append(
            total_visible_joints / num_frames
        )

        unique_track_ids_per_video.append(
            len(track_ids)
        )

    return {
        "num_samples": len(pose_files),
        "empty_samples": empty_samples,
        "mean_detection_coverage": np.mean(detection_coverages),
        "mean_tracking_coverage": np.mean(tracking_coverages),
        "mean_visible_joints_per_frame": np.mean(visible_joints_per_frame),
        "mean_keypoint_confidence": np.mean(keypoint_confidences),
        "mean_unique_track_ids_per_video": np.mean(unique_track_ids_per_video),
    }

In [ ]:
from src.config import DATASET_ROOT, POSE_DATASET_ROOT, GAMMA_POSE_DATASET_ROOT, POSE_DATASET_ROOT_OCSORT, POSE_DATASET_ROOT_BYTETRACK, POSE_DATASET_ROOT_OCSORT2 
original_results = analyse_pose_database(
    POSE_DATASET_ROOT
)

ocsort_results = analyse_pose_database(
    POSE_DATASET_ROOT_OCSORT
)

print("BYTETRACK")
for key, value in original_results.items():
    print(f"{key}: {value}")

print("OCSORT")
for key, value in ocsort_results.items():
    print(f"{key}: {value}")


BYTETRACK
num_samples: 2000
empty_samples: 288
mean_detection_coverage: 0.6915333333333334
mean_tracking_coverage: 0.6915333333333334
mean_visible_joints_per_frame: 28.619839999999996
mean_keypoint_confidence: 0.6205726204782711
mean_unique_track_ids_per_video: 6.101
OCSORT
num_samples: 2000
empty_samples: 288
mean_detection_coverage: 0.6933833333333334
mean_tracking_coverage: 0.6933833333333334
mean_visible_joints_per_frame: 28.821176666666666
mean_keypoint_confidence: 0.6197176046431078
mean_unique_track_ids_per_video: 5.7035


In [ ]:
original_results = analyse_pose_database(
    POSE_DATASET_ROOT_BYTETRACK 
)

ocsort_results = analyse_pose_database(
    POSE_DATASET_ROOT_OCSORT2
)

print("BYTETRACK")
for key, value in original_results.items():
    print(f"{key}: {value}")

print("OCSORT")
for key, value in ocsort_results.items():
    print(f"{key}: {value}")